In [1]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

# padding (16 byte üçün)
def pad(data):
    while len(data) % 16 != 0:
        data += b' '
    return data

# PRF (AES ilə)
def prf(key, r, length):
    cipher = AES.new(key, AES.MODE_ECB)
    output = b""
    counter = 0

    while len(output) < length:
        block = r + counter.to_bytes(4, 'big')
        output += cipher.encrypt(pad(block))
        counter += 1

    return output[:length]


# encrypt
def encrypt(key, message):
    message = message.encode()
    r = get_random_bytes(12)  # random nonce
    keystream = prf(key, r, len(message))
    
    cipher = bytes([m ^ k for m, k in zip(message, keystream)])
    return r, cipher


# decrypt
def decrypt(key, r, cipher):
    keystream = prf(key, r, len(cipher))
    message = bytes([c ^ k for c, k in zip(cipher, keystream)])
    return message.decode()


# test
key = get_random_bytes(16)  # AES key
msg = "Salam dunya"

r, c = encrypt(key, msg)
dec = decrypt(key, r, c)

print("Cipher:", c)
print("Decrypted:", dec)

Cipher: b'\xca\x84\xf5LIZL\x9ePZ\xb8'
Decrypted: Salam dunya
